# AO3 Tag Wrangling Analysis

Splits each seed tag's works into **literal** tag usage vs. **wrangled**
(synonym/subtag) matches, from `ao3_tag_metadata.csv`.

Browsing an AO3 canonical tag page returns works tagged with the canonical tag
OR any synonym wranglers merged into it -- but the scraped metadata records each
work's tags as the author typed them. That's why the per-field heatmaps' same-name
cells (seed tag `Angst` x displayed tag `Angst` at, say, 47.5%) are not a bug:
they measure how often authors literally typed the canonical tag. This notebook
makes that measurement explicit, plus an optional per-synonym breakdown when a
wrangling-relations CSV is available (`seed_tag, relation, related_tag` with
`relation` in `synonym`/`subtag` -- hand-built today, or from a future scraper
step).

No network access is required -- this only reads local CSVs.

In [ ]:
# Installs any of this notebook's dependencies that aren't already present.
# Safe to re-run.
%pip install -q pandas

In [ ]:
import os

import pandas as pd

## Configuration

In [ ]:
INPUT = "ao3_tag_metadata.csv"
SYNONYMS_CSV = "ao3_tag_synonyms.csv"   # wrangling relations (seed_tag, relation,
                                         # related_tag); breakdown is skipped with a
                                         # note if this file doesn't exist
LITERAL_OUT = "ao3_seed_tag_literal_usage.csv"
BREAKDOWN_OUT = "ao3_seed_tag_synonym_breakdown.csv"

DELIMITER = ", "
ALL_METADATA_FIELDS = ["rating", "warnings", "category", "fandom",
                        "relationship", "character", "additional_tags"]

## Load metadata

In [ ]:
# copied from ao3_tag_visualizer.py -- keep in sync if that file changes
def load_metadata(input_csv):
    df = pd.read_csv(input_csv, dtype=str, keep_default_na=False)
    return df


def split_values(cell, delimiter=DELIMITER):
    if not cell:
        return []
    values = [v.strip() for v in cell.split(delimiter) if v.strip()]
    return list(dict.fromkeys(values))


def explode_field(df, field):
    exploded = df[["tag", "work_id", field]].copy()
    exploded[field] = exploded[field].map(split_values)
    exploded = exploded.explode(field)
    exploded = exploded[exploded[field].notna() & (exploded[field] != "")]
    return exploded


df = load_metadata(INPUT)
df.head()

## Literal vs. wrangled split

A work is **literal** when the seed tag appears case-insensitively among any of
its displayed field values -- all seven metadata fields are checked, since a
seed tag can be a fandom/relationship/character, not just an additional_tags
value (and AO3 tags are case-insensitively unique, so a case variant is the
same tag, not a synonym). `literal_pct + wrangled_pct` always sums to 100 --
this is a strict two-way partition of each seed tag's works.

In [ ]:
# copied from ao3_tag_wrangling.py -- keep in sync if that file changes
def _literal_pairs(df):
    """(tag, work_id) pairs where the work displays its seed tag literally
    (case-insensitively) in ANY metadata field -- a seed tag can be a
    fandom/relationship/character, not just an additional_tags value."""
    matches = []
    for field in ALL_METADATA_FIELDS:
        exploded = explode_field(df, field)
        match = exploded[exploded[field].str.lower() == exploded["tag"].str.lower()]
        matches.append(match[["tag", "work_id"]])
    return pd.concat(matches, ignore_index=True).drop_duplicates()


def seed_tag_literal_usage(df):
    """Per seed tag: how many of its works display the seed tag literally
    vs. matched only via wrangling (synonyms/subtags). Returns a DataFrame
    [seed_tag, n_works, literal_works, literal_pct, wrangled_pct] sorted
    by n_works descending (tie-break: alphabetical). literal_pct +
    wrangled_pct always sums to 100 -- unlike the fandom-label outputs
    this is a strict two-way partition of each seed tag's works."""
    totals = (df.drop_duplicates(["tag", "work_id"])
                .groupby("tag")["work_id"].nunique())
    literal_counts = _literal_pairs(df).groupby("tag")["work_id"].nunique()

    result = pd.DataFrame({
        "seed_tag": totals.index,
        "n_works": totals.values,
        "literal_works": totals.index.map(literal_counts).fillna(0).astype(int),
    })
    result["literal_pct"] = (result["literal_works"] / result["n_works"] * 100).round(1)
    result["wrangled_pct"] = (100 - result["literal_pct"]).round(1)
    result = result.sort_values(["n_works", "seed_tag"], ascending=[False, True])
    return result.reset_index(drop=True)


literal_usage = seed_tag_literal_usage(df)
literal_usage.to_csv(LITERAL_OUT, index=False)
print(f"wrote {LITERAL_OUT} ({len(literal_usage)} seed tags)")
literal_usage.head(20)

## Per-synonym breakdown (optional)

Names the exact form each non-literal work used, given known wrangling
relations. A literal work counts once under `literal`; a non-literal work
counts under **every** known synonym/subtag it displays (sums may exceed 100%);
works matching nothing are `unidentified`. Skipped with a note when
`SYNONYMS_CSV` doesn't exist -- the literal-usage split above never needs it.

In [ ]:
# copied from ao3_tag_wrangling.py -- keep in sync if that file changes
def synonym_breakdown(df, relations_df):
    """Names the exact form each work used, given known wrangling
    relations. relations_df columns: seed_tag, relation, related_tag with
    relation in {synonym, subtag} (other relation values are ignored).
    Returns a DataFrame [seed_tag, matched_via, n_works, pct] where
    matched_via is "literal", the exact related-tag name, or
    "unidentified" (the work displays neither the seed tag nor any known
    relation -- e.g. a wrangling chain the relations CSV doesn't cover).
    A literal work counts once under "literal"; a non-literal work counts
    under EVERY known relation it displays (it can legitimately display
    several, so a seed tag's percentages can sum past 100 -- the same
    documented semantics as the fandom-label outputs); rows with zero
    works are omitted. pct denominator = the seed tag's total works."""
    pairs = df.drop_duplicates(["tag", "work_id"])
    totals = pairs.groupby("tag")["work_id"].nunique()

    literal = _literal_pairs(df)

    relations = relations_df[relations_df["relation"].isin(["synonym", "subtag"])].copy()
    relations["seed_lower"] = relations["seed_tag"].str.lower()
    relations["related_lower"] = relations["related_tag"].str.lower()

    matched_frames = []
    for field in ALL_METADATA_FIELDS:
        exploded = explode_field(df, field)
        exploded = exploded.assign(seed_lower=exploded["tag"].str.lower(),
                                    value_lower=exploded[field].str.lower())
        joined = exploded.merge(relations, on="seed_lower")
        joined = joined[joined["value_lower"] == joined["related_lower"]]
        matched_frames.append(joined[["tag", "work_id", "related_tag"]])
    matched = pd.concat(matched_frames, ignore_index=True).drop_duplicates()
    # "otherwise": a literal work counts once, under literal only
    # (anti-join against the literal pairs).
    matched = matched.merge(literal.assign(_literal=True),
                             on=["tag", "work_id"], how="left")
    matched = matched[matched["_literal"].isna()].drop(columns="_literal")

    rows = []
    for tag, total in totals.items():
        literal_n = literal[literal["tag"] == tag]["work_id"].nunique()
        if literal_n:
            rows.append({"seed_tag": tag, "matched_via": "literal", "n_works": literal_n})
        tag_matched = matched[matched["tag"] == tag]
        for related_tag, group in tag_matched.groupby("related_tag"):
            rows.append({"seed_tag": tag, "matched_via": related_tag,
                          "n_works": group["work_id"].nunique()})
        identified_works = set(literal[literal["tag"] == tag]["work_id"]) | \
                            set(tag_matched["work_id"])
        unidentified_n = total - len(identified_works)
        if unidentified_n:
            rows.append({"seed_tag": tag, "matched_via": "unidentified",
                          "n_works": unidentified_n})

    result = pd.DataFrame(rows, columns=["seed_tag", "matched_via", "n_works"])
    result["pct"] = (result["n_works"] / result["seed_tag"].map(totals) * 100).round(1)
    # Seed tags in the same order as seed_tag_literal_usage (n_works
    # descending, tie-break alphabetical); within a seed tag, biggest
    # slice first (tie-break: alphabetical).
    order = totals.reset_index()
    order.columns = ["tag", "n_works"]
    order = order.sort_values(["n_works", "tag"], ascending=[False, True])
    tag_order = {tag: rank for rank, tag in enumerate(order["tag"])}
    result = result.assign(_order=result["seed_tag"].map(tag_order))
    result = result.sort_values(["_order", "n_works", "matched_via"],
                                 ascending=[True, False, True]).drop(columns="_order")
    return result.reset_index(drop=True)


breakdown = None
if os.path.exists(SYNONYMS_CSV):
    relations_df = pd.read_csv(SYNONYMS_CSV, dtype=str, keep_default_na=False)
    missing = {"seed_tag", "relation", "related_tag"} - set(relations_df.columns)
    assert not missing, f"{SYNONYMS_CSV} is missing columns {sorted(missing)}"
    breakdown = synonym_breakdown(df, relations_df)
    breakdown.to_csv(BREAKDOWN_OUT, index=False)
    print(f"wrote {BREAKDOWN_OUT} ({len(breakdown)} rows)")
else:
    print(f"note: {SYNONYMS_CSV} not found -- skipping the per-synonym breakdown")

breakdown.head(30) if breakdown is not None else None

## Done

`{LITERAL_OUT}` (and `{BREAKDOWN_OUT}`, when `{SYNONYMS_CSV}` exists) are now in
the notebook's working directory. Re-run from the Configuration cell with
different paths to reanalyze.